# NumCompute Stream Demo

This notebook demonstrates the Assignment 2.2 streaming workflow: CSV loading through the custom I/O module, chunk-wise incremental training, metric logging, model comparison, and visualisation.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'numcompute_stream').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from numcompute_stream.datasets import iter_chunks
from numcompute_stream.ensemble import EnsembleClassifier
from numcompute_stream.io import load_csv, make_classification_data, save_csv, train_test_split
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.preprocessing import SimpleImputer, StandardScaler
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.visualise import compare_models, plot_metric_over_time, plot_predictions_vs_ground_truth

DATA_PATH = PROJECT_ROOT / 'demo' / 'stream_demo_data.csv'
FIGURE_DIR = PROJECT_ROOT / 'demo' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
if not DATA_PATH.exists():
    X_raw, y_raw = make_classification_data(n_samples=600, n_features=6, n_classes=2, random_state=42, noise=0.45)
    header = ','.join([f'feature_{i}' for i in range(X_raw.shape[1])] + ['target'])
    save_csv(DATA_PATH, X_raw, y_raw, header=header)

X, y = load_csv(DATA_PATH, skip_header=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=11)
X_train.shape, X_test.shape

In [ ]:
def make_tree_pipeline():
    return Pipeline([
        ('impute', SimpleImputer(strategy='mean')),
        ('scale', StandardScaler()),
        ('model', DecisionTreeClassifier(max_depth=5, random_state=7)),
    ])

def make_ensemble_pipeline():
    return Pipeline([
        ('impute', SimpleImputer(strategy='mean')),
        ('scale', StandardScaler()),
        ('model', EnsembleClassifier(n_estimators=7, method='random_forest', max_depth=5, random_state=7)),
    ])

def train_stream(pipe, X_train, y_train, chunk_size=50):
    trainer = StreamTrainer(pipe)
    for X_chunk, y_chunk in iter_chunks(X_train, y_train, chunk_size=chunk_size):
        trainer.fit_chunk(X_chunk, y_chunk)
    return trainer

tree_trainer = train_stream(make_tree_pipeline(), X_train, y_train)
ensemble_trainer = train_stream(make_ensemble_pipeline(), X_train, y_train)

tree_score = tree_trainer.model.score(X_test, y_test)
ensemble_score = ensemble_trainer.model.score(X_test, y_test)
tree_score, ensemble_score

In [ ]:
tree_accuracy = tree_trainer.metric_values('chunk_accuracy')
ensemble_accuracy = ensemble_trainer.metric_values('chunk_accuracy')

plot_metric_over_time(ensemble_accuracy, title='Streaming ensemble accuracy', ylabel='Chunk accuracy')
compare_models(tree_accuracy, ensemble_accuracy, labels=('Decision tree', 'Random forest'), title='Streaming model comparison', ylabel='Chunk accuracy')

y_pred = ensemble_trainer.model.predict(X_test)
plot_predictions_vs_ground_truth(y_test[:80], y_pred[:80], title='Predictions vs ground truth')

In [ ]:
ensemble_trainer.logs_[:3]